# Speaker Diarization Exploration
Steps:
1. Run NVIDIA NeMo Sortformer Diarizer portion in SageMaker (see below) on .wav files and copy timestamp output. You will need a hugging face token. NeMo performance is stronger with noisy data, and fast with GPU but still makes mistakes.
2. OPTIONAL - run pyannote diarizer on local CPU for comparison, but performance is poor on noisy files. You will need a hugging face token, and to request access to several pyannote models listed below.
3. Reformat output to work with front-end that will display timestamps: [{"start": 0.031, "end": 1.027, "content": "SPEAKER_07"}, ...]
4. Reformat audio for age classification. Iterates through output and for each speaker, splits the audio into separate files for each speaker. 
5. Run these individual speaker files through age classifier (see separate repo/notebook in sagemaker) and identify age. Fill out dictionary to re-map labels in this format: 
label_dict = {
  "speaker_0": "adult_0",
  "speaker_1": "adult_1",...
}
6. Modify the labels using the label dictionary in the last section of this program. 
7. Copy/paste output into speech sandbox Custom tab to view timestamps on the audio

Author: K Rubio

Last Updated: July 2026


## NVIDIA NeMo Sortformer Diarizer
Run on SageMaker with instance ml.g5.xlarge and 5GB of storage

Documentation: https://docs.nvidia.com/nemo-framework/user-guide/latest/nemotoolkit/asr/speaker_diarization/models.html#sortformer-diarizer 

Tutorial: https://github.com/NVIDIA-NeMo/Speech/blob/main/tutorials/speaker_tasks/End_to_End_Diarization_Inference.ipynb

In [ ]:
!pip install wget
!pip install sox libsndfile ffmpeg
!pip install text-unidecode
!pip install ipython

In [ ]:
BRANCH = 'main'
!python -m pip install "nemo_toolkit[asr] @ git+https://github.com/NVIDIA/NeMo.git@{BRANCH}"

In [ ]:
import time
from nemo.collections.asr.models import SortformerEncLabelModel
from huggingface_hub import get_token as get_hf_token
import torch
import json
import ast
import os

In [ ]:
os.environ["HF_TOKEN"] = "hf_XXXXXX" # replace with HF token

if get_hf_token() is not None and get_hf_token().startswith("hf_"):
    # If you have logged into HuggingFace hub and have access token
    diar_model = SortformerEncLabelModel.from_pretrained("nvidia/diar_sortformer_4spk-v1")
else:
    # You can downloaded ".nemo" file from https://huggingface.co/nvidia/diar_sortformer_4spk-v1 and specify the path.
    diar_model = SortformerEncLabelModel.restore_from(restore_path="/path/to/diar_sortformer_4spk-v1.nemo", map_location=torch.device('cuda'), strict=False)


In [ ]:
filename = 'eastt.wav'
start = time.time()
pred_list, pred_tensor_list = diar_model.diarize(audio=filename, include_tensor_outputs=True)

In [ ]:

print(pred_list[0])
print(pred_tensor_list[0].shape)
end = time.time()
print('Diarization took ', round((end - start), 2), 'seconds')


In [ ]:

# Reformat NeMo output
# Takes in data_str = "['0.880 2.640 speaker_0', '2.880 8.400 speaker_1']"
# Outputs/prints this format: [{"start": 0.880, "end": 2.640, "content": "speaker_0"}, {"start": 2.880, "end": 8.400, "content": "speaker_1"}]
def reformat(data_str):
    lines = ast.literal_eval(data_str)  # safely parses the string into a real list
    result = []
    for line in lines:
        start, end, content = line.split()
        result.append({
            "start": float(start),
            "end": float(end),
            "content": content
        })
    print(json.dumps(result))
    return result

data = "['295.327 295.715 SPEAKER_04']"
reformat(data)

In [ ]:
print(json.dumps(data))

## Pyannote speaker diarization
Run on CPU locally
Can theoretically run faster on GPU, but couldn't resolve dependency issues with torch/torch.audio version in sagemaker
Time: will take almost the duration of the audio

In [2]:
!pip install pyannote.audio

Looking in indexes: https://nexus.cainc.com/repository/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.6/894.6 kB 918.1 kB/s eta 0:00:00a 0:00:01
  Using cached https://nexus.cainc.com/repository/pypi/packages/opentelemetry-api/1.43.0/opentelemetry_api-1.43.0-py3-none-any.whl (61 kB)
  Using cached https://nexus.cainc.com/repository/pypi/packages/opentelemetry-sdk/1.43.0/opentelemetry_sdk-1.43.0-py3-none-any.whl (178 kB)
  Using cached https://nexus.cainc.com/repository/pypi/packages/rich/15.0.0/rich-15.0.0-py3-none-any.whl (310 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 2.3 MB/s eta 0:00:00a 0:00:01
  Using cached https://nexus.cainc.com/repository/pypi/packages/googleapis-common-protos/1.75.0/googleapis_common_protos-1.75.0-py3-none-any.whl (300 kB)
  Using cached https://nexus.cainc.com/repository/pypi/packages/opentelemetry-proto/1.43.0/opentelemetry_proto-1.43.0-py3-none-any.whl (72 kB)
  Using cached https://nexus.cainc.com/repository/pypi/packages/op

In [ ]:
from pyannote.audio import Pipeline
import torch
import json

/Users/KRubio/CA_Repos/kays_repos/audio_quality_classification/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [51]:
def parse_segments(lines):
  """
  Convert lines like '0.031 1.027 SPEAKER_07' into a list of dicts:
  [{"start": 0.031, "end": 1.027, "content": "SPEAKER_07"}, ...]
  """
  result = []
  for line in lines:
    line = line.strip()
    if not line:
      continue
    start, end, content = line.split()
    result.append({
      "start": float(start),
      "end": float(end),
      "content": content
    })
  return result

In [ ]:
# go to each of these sites and fill out the form to access the model, then copy any one of your account tokens below
# that has in its permissions "Read access to contents of all public gated repos you can access"
# https://huggingface.co/pyannote/speaker-diarization-3.1
# https://huggingface.co/pyannote/segmentation-3.0
# https://huggingface.co/pyannote/speaker-diarization-community-1
pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    token="hf_XXXXX"
)

In [48]:
filename = 'katie_clipped_5-10min'
filepath = f'../data/audio/CHILDES/{filename}.wav'
diarization = pipeline(filepath)


In [55]:
output = []
for turn, speaker in diarization.speaker_diarization:
    #print(f"'{turn.start:.3f} {turn.end:.3f} {speaker}',")
    output.append({
        "start": round(turn.start, 3),
        "end": round(turn.end, 3),
        "content": speaker
    })
#print(output)
print(json.dumps(output))

[{"start": 0.031, "end": 1.027, "content": "SPEAKER_07"}, {"start": 1.988, "end": 5.937, "content": "SPEAKER_07"}, {"start": 6.393, "end": 6.899, "content": "SPEAKER_07"}, {"start": 6.899, "end": 8.536, "content": "SPEAKER_05"}, {"start": 8.283, "end": 12.974, "content": "SPEAKER_07"}, {"start": 8.688, "end": 9.97, "content": "SPEAKER_05"}, {"start": 13.987, "end": 14.712, "content": "SPEAKER_07"}, {"start": 14.78, "end": 17.767, "content": "SPEAKER_07"}, {"start": 17.952, "end": 18.273, "content": "SPEAKER_07"}, {"start": 18.357, "end": 24.972, "content": "SPEAKER_07"}, {"start": 26.626, "end": 27.588, "content": "SPEAKER_07"}, {"start": 30.71, "end": 32.498, "content": "SPEAKER_07"}, {"start": 32.65, "end": 33.055, "content": "SPEAKER_07"}, {"start": 37.595, "end": 40.632, "content": "SPEAKER_06"}, {"start": 41.088, "end": 42.573, "content": "SPEAKER_01"}, {"start": 42.607, "end": 43.855, "content": "SPEAKER_06"}, {"start": 44.277, "end": 44.699, "content": "SPEAKER_06"}, {"start": 4

In [ ]:
# another way to print that's more easily readable
for turn, speaker in diarization.speaker_diarization:
    print(f"start: '{turn.start:.3f}, end: {turn.end:.3f}, content: {speaker}',")


start: '0.031, end: 1.027, content: SPEAKER_07',
start: '1.988, end: 5.937, content: SPEAKER_07',
start: '6.393, end: 6.899, content: SPEAKER_07',
start: '6.899, end: 8.536, content: SPEAKER_05',
start: '8.283, end: 12.974, content: SPEAKER_07',
start: '8.688, end: 9.970, content: SPEAKER_05',
start: '13.987, end: 14.712, content: SPEAKER_07',
start: '14.780, end: 17.767, content: SPEAKER_07',
start: '17.952, end: 18.273, content: SPEAKER_07',
start: '18.357, end: 24.972, content: SPEAKER_07',
start: '26.626, end: 27.588, content: SPEAKER_07',
start: '30.710, end: 32.498, content: SPEAKER_07',
start: '32.650, end: 33.055, content: SPEAKER_07',
start: '37.595, end: 40.632, content: SPEAKER_06',
start: '41.088, end: 42.573, content: SPEAKER_01',
start: '42.607, end: 43.855, content: SPEAKER_06',
start: '44.277, end: 44.699, content: SPEAKER_06',
start: '44.665, end: 45.357, content: SPEAKER_01',
start: '46.066, end: 46.876, content: SPEAKER_06',
start: '47.365, end: 48.631, content: SPEA

In [59]:
data = "['0.880 2.640 speaker_0', '2.880 8.400 speaker_1']"
#data2 = [{"start": 0.880, "end": 2.640, "content": "speaker_0"}, {"start": 2.880, "end": 8.400, "content": "speaker_1"}]

## Reformat NeMo output

In [ ]:
import ast
import json

In [1]:
# Takes in data_str = "['0.880 2.640 speaker_0', '2.880 8.400 speaker_1']"
# Outputs/prints this format: [{"start": 0.880, "end": 2.640, "content": "speaker_0"}, {"start": 2.880, "end": 8.400, "content": "speaker_1"}]
def reformat(data_str):
    lines = ast.literal_eval(data_str)  # safely parses the string into a real list
    result = []
    for line in lines:
        start, end, content = line.split()
        result.append({
            "start": float(start),
            "end": float(end),
            "content": content
        })
    print(json.dumps(result))
    return result

In [56]:
data = "['0.031 1.027 SPEAKER_07', '1.988 5.937 SPEAKER_07', '6.393 6.899 SPEAKER_07', '6.899 8.536 SPEAKER_05', '8.283 12.974 SPEAKER_07', '8.688 9.970 SPEAKER_05', '13.987 14.712 SPEAKER_07', '14.780 17.767 SPEAKER_07', '17.952 18.273 SPEAKER_07', '18.357 24.972 SPEAKER_07', '26.626 27.588 SPEAKER_07', '30.710 32.498 SPEAKER_07', '32.650 33.055 SPEAKER_07', '37.595 40.632 SPEAKER_06', '41.088 42.573 SPEAKER_01', '42.607 43.855 SPEAKER_06', '44.277 44.699 SPEAKER_06', '44.665 45.357 SPEAKER_01', '46.066 46.876 SPEAKER_06', '47.365 48.631 SPEAKER_01', '48.631 48.682 SPEAKER_02', '51.027 51.533 SPEAKER_02', '51.635 53.153 SPEAKER_06', '53.491 55.195 SPEAKER_02', '55.617 56.731 SPEAKER_06', '57.018 57.710 SPEAKER_02', '57.507 59.600 SPEAKER_06', '60.275 61.186 SPEAKER_02', '61.338 61.372 SPEAKER_02', '61.372 62.722 SPEAKER_06', '63.278 64.190 SPEAKER_04', '63.987 65.033 SPEAKER_06', '65.033 65.202 SPEAKER_05', '65.405 66.265 SPEAKER_04', '66.518 68.864 SPEAKER_04', '68.948 69.995 SPEAKER_05', '70.703 72.138 SPEAKER_05', '73.083 73.117 SPEAKER_04', '73.117 74.669 SPEAKER_00', '74.669 75.277 SPEAKER_05', '74.686 74.703 SPEAKER_00', '75.277 75.293 SPEAKER_06', '75.766 76.373 SPEAKER_00', '76.289 76.340 SPEAKER_05', '76.340 76.357 SPEAKER_06', '76.373 76.863 SPEAKER_06', '76.863 76.964 SPEAKER_05', '76.964 77.588 SPEAKER_06', '77.825 80.238 SPEAKER_00', '80.980 81.436 SPEAKER_06', '82.364 83.005 SPEAKER_00', '85.098 85.655 SPEAKER_06', '85.942 87.477 SPEAKER_04', '88.540 88.557 SPEAKER_06', '88.557 88.574 SPEAKER_05', '88.574 88.962 SPEAKER_06', '88.962 89.165 SPEAKER_05', '90.886 91.797 SPEAKER_04', '91.831 92.337 SPEAKER_05', '93.400 95.054 SPEAKER_05', '96.269 97.265 SPEAKER_05', '100.977 100.994 SPEAKER_05', '100.994 102.665 SPEAKER_06', '101.095 101.146 SPEAKER_01', '101.146 101.163 SPEAKER_00', '101.163 102.175 SPEAKER_01', '103.677 104.302 SPEAKER_06', '103.846 103.947 SPEAKER_04', '103.947 104.032 SPEAKER_01', '104.032 104.268 SPEAKER_04', '104.302 104.572 SPEAKER_04', '104.572 104.622 SPEAKER_06', '104.622 104.774 SPEAKER_04', '104.774 104.825 SPEAKER_06', '104.875 105.618 SPEAKER_06', '106.107 106.833 SPEAKER_04', '106.225 106.681 SPEAKER_06', '106.833 106.867 SPEAKER_06', '106.968 108.487 SPEAKER_06', '108.723 110.444 SPEAKER_04', '110.410 110.815 SPEAKER_05', '111.288 113.617 SPEAKER_05', '114.359 114.376 SPEAKER_04', '114.376 114.578 SPEAKER_03', '114.578 116.603 SPEAKER_05', '116.435 117.818 SPEAKER_03', '117.970 118.443 SPEAKER_05', '118.814 120.417 SPEAKER_03', '121.244 121.649 SPEAKER_03', '122.088 123.590 SPEAKER_03', '126.307 127.117 SPEAKER_05', '128.956 130.846 SPEAKER_05', '136.465 139.435 SPEAKER_02', '140.043 141.477 SPEAKER_05', '141.983 142.422 SPEAKER_02', '142.422 142.439 SPEAKER_01', '142.608 143.738 SPEAKER_05', '143.924 144.700 SPEAKER_01', '145.510 147.215 SPEAKER_05', '148.244 148.700 SPEAKER_01', '149.020 149.611 SPEAKER_05', '149.797 151.535 SPEAKER_01', '151.973 153.138 SPEAKER_01', '155.551 156.462 SPEAKER_05', '157.795 157.964 SPEAKER_01', '157.964 158.825 SPEAKER_04', '158.825 158.842 SPEAKER_01', '158.993 159.196 SPEAKER_05', '159.685 160.411 SPEAKER_05', '161.052 161.879 SPEAKER_04', '162.233 163.769 SPEAKER_05', '163.820 164.545 SPEAKER_05', '167.735 168.595 SPEAKER_05', '168.595 168.697 SPEAKER_06', '169.085 170.131 SPEAKER_00', '170.131 170.485 SPEAKER_06', '170.738 171.970 SPEAKER_00', '172.240 173.641 SPEAKER_06', '174.367 175.295 SPEAKER_00', '176.746 179.429 SPEAKER_00', '179.547 180.965 SPEAKER_06', '181.690 182.652 SPEAKER_02', '183.158 183.530 SPEAKER_06', '183.749 185.150 SPEAKER_02', '185.605 187.748 SPEAKER_02', '188.305 191.107 SPEAKER_02', '191.107 191.562 SPEAKER_06', '194.498 195.224 SPEAKER_06', '195.528 195.933 SPEAKER_02', '196.237 196.760 SPEAKER_06', '197.367 198.548 SPEAKER_02', '200.827 201.029 SPEAKER_06', '201.029 201.417 SPEAKER_05', '201.637 202.463 SPEAKER_02', '203.898 204.016 SPEAKER_02', '204.016 205.822 SPEAKER_03', '206.615 208.268 SPEAKER_03', '208.707 208.994 SPEAKER_03', '208.994 209.112 SPEAKER_06', '209.112 210.327 SPEAKER_03', '210.327 210.563 SPEAKER_06', '211.930 213.432 SPEAKER_06', '213.635 213.753 SPEAKER_03', '213.753 214.124 SPEAKER_02', '214.124 216.200 SPEAKER_06', '216.318 216.335 SPEAKER_02', '216.352 216.689 SPEAKER_02', '217.043 218.157 SPEAKER_02', '219.187 219.355 SPEAKER_06', '220.098 221.431 SPEAKER_06', '222.055 222.477 SPEAKER_04', '224.148 225.413 SPEAKER_06', '226.004 226.595 SPEAKER_04', '227.877 230.155 SPEAKER_04', '232.568 234.442 SPEAKER_05', '235.977 237.192 SPEAKER_04', '239.892 241.664 SPEAKER_04', '242.592 244.533 SPEAKER_04', '244.347 247.132 SPEAKER_05', '247.655 248.802 SPEAKER_04', '252.616 253.173 SPEAKER_04', '253.173 253.763 SPEAKER_01', '253.763 253.932 SPEAKER_04', '254.202 255.873 SPEAKER_05', '257.476 258.218 SPEAKER_01', '258.792 259.990 SPEAKER_01', '260.193 262.606 SPEAKER_05', '263.888 265.205 SPEAKER_05', '266.926 267.719 SPEAKER_05', '267.719 267.770 SPEAKER_06', '268.715 269.272 SPEAKER_04', '269.086 270.773 SPEAKER_06', '271.128 272.748 SPEAKER_04', '272.933 273.777 SPEAKER_06', '273.777 273.811 SPEAKER_05', '274.958 275.735 SPEAKER_05', '276.022 276.055 SPEAKER_04', '276.055 276.832 SPEAKER_00', '277.169 277.203 SPEAKER_04', '277.203 278.452 SPEAKER_00', '278.249 279.481 SPEAKER_05', '279.970 281.793 SPEAKER_00', '282.502 283.092 SPEAKER_00', '283.092 283.463 SPEAKER_05', '284.003 284.611 SPEAKER_05', '285.016 285.438 SPEAKER_00', '285.826 286.265 SPEAKER_05', '287.294 288.965 SPEAKER_04', '289.370 290.652 SPEAKER_04', '291.597 292.559 SPEAKER_04', '292.998 294.584 SPEAKER_05', '294.635 295.073 SPEAKER_05', '295.327 295.715 SPEAKER_04']"

In [57]:
reformat(data)

[{"start": 0.031, "end": 1.027, "content": "SPEAKER_07"}, {"start": 1.988, "end": 5.937, "content": "SPEAKER_07"}, {"start": 6.393, "end": 6.899, "content": "SPEAKER_07"}, {"start": 6.899, "end": 8.536, "content": "SPEAKER_05"}, {"start": 8.283, "end": 12.974, "content": "SPEAKER_07"}, {"start": 8.688, "end": 9.97, "content": "SPEAKER_05"}, {"start": 13.987, "end": 14.712, "content": "SPEAKER_07"}, {"start": 14.78, "end": 17.767, "content": "SPEAKER_07"}, {"start": 17.952, "end": 18.273, "content": "SPEAKER_07"}, {"start": 18.357, "end": 24.972, "content": "SPEAKER_07"}, {"start": 26.626, "end": 27.588, "content": "SPEAKER_07"}, {"start": 30.71, "end": 32.498, "content": "SPEAKER_07"}, {"start": 32.65, "end": 33.055, "content": "SPEAKER_07"}, {"start": 37.595, "end": 40.632, "content": "SPEAKER_06"}, {"start": 41.088, "end": 42.573, "content": "SPEAKER_01"}, {"start": 42.607, "end": 43.855, "content": "SPEAKER_06"}, {"start": 44.277, "end": 44.699, "content": "SPEAKER_06"}, {"start": 4

[{'start': 0.031, 'end': 1.027, 'content': 'SPEAKER_07'},
 {'start': 1.988, 'end': 5.937, 'content': 'SPEAKER_07'},
 {'start': 6.393, 'end': 6.899, 'content': 'SPEAKER_07'},
 {'start': 6.899, 'end': 8.536, 'content': 'SPEAKER_05'},
 {'start': 8.283, 'end': 12.974, 'content': 'SPEAKER_07'},
 {'start': 8.688, 'end': 9.97, 'content': 'SPEAKER_05'},
 {'start': 13.987, 'end': 14.712, 'content': 'SPEAKER_07'},
 {'start': 14.78, 'end': 17.767, 'content': 'SPEAKER_07'},
 {'start': 17.952, 'end': 18.273, 'content': 'SPEAKER_07'},
 {'start': 18.357, 'end': 24.972, 'content': 'SPEAKER_07'},
 {'start': 26.626, 'end': 27.588, 'content': 'SPEAKER_07'},
 {'start': 30.71, 'end': 32.498, 'content': 'SPEAKER_07'},
 {'start': 32.65, 'end': 33.055, 'content': 'SPEAKER_07'},
 {'start': 37.595, 'end': 40.632, 'content': 'SPEAKER_06'},
 {'start': 41.088, 'end': 42.573, 'content': 'SPEAKER_01'},
 {'start': 42.607, 'end': 43.855, 'content': 'SPEAKER_06'},
 {'start': 44.277, 'end': 44.699, 'content': 'SPEAKER_0

## Reformat audio for age classification

In [71]:
import librosa
import numpy as np
import soundfile as sf

In [194]:
data = [{"start": 0.0, "end": 1.2, "content": "speaker_0"}, {"start": 2.4, "end": 3.52, "content": "speaker_0"}, {"start": 3.76, "end": 4.56, "content": "speaker_0"}, {"start": 4.64, "end": 5.6, "content": "speaker_0"}, {"start": 6.56, "end": 6.8, "content": "speaker_0"}, {"start": 8.32, "end": 9.92, "content": "speaker_0"}, {"start": 10.08, "end": 12.8, "content": "speaker_0"}, {"start": 13.52, "end": 13.68, "content": "speaker_0"}, {"start": 14.08, "end": 14.64, "content": "speaker_0"}, {"start": 14.96, "end": 17.76, "content": "speaker_0"}, {"start": 18.08, "end": 18.32, "content": "speaker_0"}, {"start": 18.48, "end": 19.92, "content": "speaker_0"}, {"start": 20.16, "end": 22.4, "content": "speaker_0"}, {"start": 22.88, "end": 24.72, "content": "speaker_0"}, {"start": 25.52, "end": 25.76, "content": "speaker_0"}, {"start": 26.72, "end": 27.6, "content": "speaker_0"}, {"start": 30.8, "end": 31.92, "content": "speaker_0"}, {"start": 32.72, "end": 33.04, "content": "speaker_0"}, {"start": 7.2, "end": 8.24, "content": "speaker_1"}, {"start": 37.68, "end": 40.64, "content": "speaker_1"}, {"start": 42.64, "end": 43.92, "content": "speaker_1"}, {"start": 44.32, "end": 44.88, "content": "speaker_1"}, {"start": 46.16, "end": 46.8, "content": "speaker_1"}, {"start": 51.6, "end": 53.2, "content": "speaker_1"}, {"start": 55.68, "end": 56.56, "content": "speaker_1"}, {"start": 57.52, "end": 59.68, "content": "speaker_1"}, {"start": 61.36, "end": 61.76, "content": "speaker_1"}, {"start": 61.92, "end": 62.8, "content": "speaker_1"}, {"start": 64.16, "end": 65.2, "content": "speaker_1"}, {"start": 66.64, "end": 66.8, "content": "speaker_1"}, {"start": 68.96, "end": 70.0, "content": "speaker_1"}, {"start": 70.8, "end": 72.16, "content": "speaker_1"}, {"start": 74.64, "end": 75.44, "content": "speaker_1"}, {"start": 76.32, "end": 77.52, "content": "speaker_1"}, {"start": 80.0, "end": 80.32, "content": "speaker_1"}, {"start": 84.24, "end": 84.48, "content": "speaker_1"}, {"start": 85.04, "end": 85.68, "content": "speaker_1"}, {"start": 88.0, "end": 89.2, "content": "speaker_1"}, {"start": 90.24, "end": 90.48, "content": "speaker_1"}, {"start": 91.92, "end": 92.32, "content": "speaker_1"}, {"start": 93.6, "end": 94.72, "content": "speaker_1"}, {"start": 96.4, "end": 97.04, "content": "speaker_1"}, {"start": 101.04, "end": 102.64, "content": "speaker_1"}, {"start": 103.84, "end": 104.24, "content": "speaker_1"}, {"start": 104.96, "end": 105.6, "content": "speaker_1"}, {"start": 107.04, "end": 108.32, "content": "speaker_1"}, {"start": 110.4, "end": 110.8, "content": "speaker_1"}, {"start": 111.28, "end": 113.52, "content": "speaker_1"}, {"start": 114.56, "end": 116.64, "content": "speaker_1"}, {"start": 118.0, "end": 118.48, "content": "speaker_1"}, {"start": 121.2, "end": 121.76, "content": "speaker_1"}, {"start": 123.36, "end": 124.64, "content": "speaker_1"}, {"start": 126.16, "end": 127.2, "content": "speaker_1"}, {"start": 129.04, "end": 130.56, "content": "speaker_1"}, {"start": 133.2, "end": 133.6, "content": "speaker_1"}, {"start": 140.08, "end": 141.44, "content": "speaker_1"}, {"start": 142.72, "end": 143.68, "content": "speaker_1"}, {"start": 145.52, "end": 147.2, "content": "speaker_1"}, {"start": 149.04, "end": 149.68, "content": "speaker_1"}, {"start": 151.76, "end": 152.4, "content": "speaker_1"}, {"start": 155.6, "end": 156.48, "content": "speaker_1"}, {"start": 159.04, "end": 159.28, "content": "speaker_1"}, {"start": 159.76, "end": 160.48, "content": "speaker_1"}, {"start": 162.24, "end": 163.68, "content": "speaker_1"}, {"start": 163.84, "end": 163.92, "content": "speaker_1"}, {"start": 164.16, "end": 164.56, "content": "speaker_1"}, {"start": 167.76, "end": 168.64, "content": "speaker_1"}, {"start": 170.16, "end": 170.48, "content": "speaker_1"}, {"start": 172.24, "end": 173.6, "content": "speaker_1"}, {"start": 179.6, "end": 180.96, "content": "speaker_1"}, {"start": 183.2, "end": 183.52, "content": "speaker_1"}, {"start": 187.76, "end": 187.84, "content": "speaker_1"}, {"start": 190.88, "end": 191.52, "content": "speaker_1"}, {"start": 194.56, "end": 195.2, "content": "speaker_1"}, {"start": 196.32, "end": 196.72, "content": "speaker_1"}, {"start": 200.88, "end": 201.44, "content": "speaker_1"}, {"start": 210.32, "end": 210.64, "content": "speaker_1"}, {"start": 212.0, "end": 213.28, "content": "speaker_1"}, {"start": 214.32, "end": 214.4, "content": "speaker_1"}, {"start": 214.8, "end": 216.08, "content": "speaker_1"}, {"start": 219.12, "end": 219.44, "content": "speaker_1"}, {"start": 220.16, "end": 221.44, "content": "speaker_1"}, {"start": 224.16, "end": 225.36, "content": "speaker_1"}, {"start": 232.56, "end": 234.4, "content": "speaker_1"}, {"start": 244.48, "end": 246.24, "content": "speaker_1"}, {"start": 246.32, "end": 247.12, "content": "speaker_1"}, {"start": 253.68, "end": 254.08, "content": "speaker_1"}, {"start": 254.16, "end": 255.92, "content": "speaker_1"}, {"start": 260.24, "end": 262.64, "content": "speaker_1"}, {"start": 263.92, "end": 265.28, "content": "speaker_1"}, {"start": 266.96, "end": 267.84, "content": "speaker_1"}, {"start": 269.04, "end": 270.8, "content": "speaker_1"}, {"start": 272.96, "end": 273.84, "content": "speaker_1"}, {"start": 275.04, "end": 275.76, "content": "speaker_1"}, {"start": 278.32, "end": 279.52, "content": "speaker_1"}, {"start": 282.56, "end": 283.04, "content": "speaker_1"}, {"start": 284.16, "end": 284.64, "content": "speaker_1"}, {"start": 285.92, "end": 286.24, "content": "speaker_1"}, {"start": 293.12, "end": 294.56, "content": "speaker_1"}, {"start": 294.8, "end": 295.12, "content": "speaker_1"}, {"start": 41.2, "end": 42.56, "content": "speaker_2"}, {"start": 44.72, "end": 45.28, "content": "speaker_2"}, {"start": 47.44, "end": 48.64, "content": "speaker_2"}, {"start": 51.2, "end": 51.44, "content": "speaker_2"}, {"start": 53.6, "end": 53.92, "content": "speaker_2"}, {"start": 54.0, "end": 55.2, "content": "speaker_2"}, {"start": 57.2, "end": 57.68, "content": "speaker_2"}, {"start": 60.72, "end": 61.2, "content": "speaker_2"}, {"start": 63.68, "end": 64.32, "content": "speaker_2"}, {"start": 65.52, "end": 66.24, "content": "speaker_2"}, {"start": 66.64, "end": 68.88, "content": "speaker_2"}, {"start": 73.2, "end": 74.32, "content": "speaker_2"}, {"start": 74.4, "end": 74.64, "content": "speaker_2"}, {"start": 75.92, "end": 76.32, "content": "speaker_2"}, {"start": 77.92, "end": 79.12, "content": "speaker_2"}, {"start": 79.44, "end": 79.76, "content": "speaker_2"}, {"start": 81.2, "end": 81.28, "content": "speaker_2"}, {"start": 86.0, "end": 86.56, "content": "speaker_2"}, {"start": 90.96, "end": 91.76, "content": "speaker_2"}, {"start": 101.36, "end": 102.0, "content": "speaker_2"}, {"start": 103.76, "end": 103.92, "content": "speaker_2"}, {"start": 104.08, "end": 104.8, "content": "speaker_2"}, {"start": 106.24, "end": 106.8, "content": "speaker_2"}, {"start": 108.8, "end": 110.48, "content": "speaker_2"}, {"start": 114.48, "end": 114.72, "content": "speaker_2"}, {"start": 116.56, "end": 117.76, "content": "speaker_2"}, {"start": 118.88, "end": 120.72, "content": "speaker_2"}, {"start": 122.16, "end": 123.44, "content": "speaker_2"}, {"start": 136.56, "end": 137.84, "content": "speaker_2"}, {"start": 138.16, "end": 139.52, "content": "speaker_2"}, {"start": 142.0, "end": 142.32, "content": "speaker_2"}, {"start": 144.0, "end": 144.32, "content": "speaker_2"}, {"start": 148.32, "end": 148.72, "content": "speaker_2"}, {"start": 149.84, "end": 151.6, "content": "speaker_2"}, {"start": 152.16, "end": 153.2, "content": "speaker_2"}, {"start": 157.84, "end": 158.88, "content": "speaker_2"}, {"start": 161.12, "end": 161.92, "content": "speaker_2"}, {"start": 169.2, "end": 169.6, "content": "speaker_2"}, {"start": 170.8, "end": 172.0, "content": "speaker_2"}, {"start": 174.4, "end": 175.36, "content": "speaker_2"}, {"start": 176.8, "end": 179.44, "content": "speaker_2"}, {"start": 181.76, "end": 182.72, "content": "speaker_2"}, {"start": 183.36, "end": 183.52, "content": "speaker_2"}, {"start": 183.84, "end": 185.2, "content": "speaker_2"}, {"start": 185.68, "end": 187.68, "content": "speaker_2"}, {"start": 188.4, "end": 189.36, "content": "speaker_2"}, {"start": 189.52, "end": 190.72, "content": "speaker_2"}, {"start": 195.68, "end": 195.84, "content": "speaker_2"}, {"start": 197.44, "end": 198.56, "content": "speaker_2"}, {"start": 201.68, "end": 202.56, "content": "speaker_2"}, {"start": 204.0, "end": 205.84, "content": "speaker_2"}, {"start": 206.72, "end": 208.32, "content": "speaker_2"}, {"start": 208.8, "end": 210.16, "content": "speaker_2"}, {"start": 213.68, "end": 214.24, "content": "speaker_2"}, {"start": 217.12, "end": 218.24, "content": "speaker_2"}, {"start": 222.16, "end": 222.56, "content": "speaker_2"}, {"start": 226.16, "end": 226.56, "content": "speaker_2"}, {"start": 227.92, "end": 230.16, "content": "speaker_2"}, {"start": 236.0, "end": 237.28, "content": "speaker_2"}, {"start": 239.92, "end": 241.76, "content": "speaker_2"}, {"start": 242.64, "end": 244.56, "content": "speaker_2"}, {"start": 247.76, "end": 248.96, "content": "speaker_2"}, {"start": 252.56, "end": 253.76, "content": "speaker_2"}, {"start": 257.52, "end": 258.4, "content": "speaker_2"}, {"start": 258.8, "end": 260.08, "content": "speaker_2"}, {"start": 268.8, "end": 269.36, "content": "speaker_2"}, {"start": 271.2, "end": 272.8, "content": "speaker_2"}, {"start": 276.08, "end": 276.88, "content": "speaker_2"}, {"start": 277.2, "end": 278.4, "content": "speaker_2"}, {"start": 280.08, "end": 281.92, "content": "speaker_2"}, {"start": 285.12, "end": 285.52, "content": "speaker_2"}, {"start": 287.36, "end": 289.12, "content": "speaker_2"}, {"start": 289.44, "end": 290.72, "content": "speaker_2"}, {"start": 291.68, "end": 292.56, "content": "speaker_2"}, {"start": 295.44, "end": 295.76, "content": "speaker_2"}, {"start": 60.4, "end": 60.72, "content": "speaker_3"}, {"start": 63.44, "end": 63.52, "content": "speaker_3"}, {"start": 74.24, "end": 74.4, "content": "speaker_3"}, {"start": 79.12, "end": 79.52, "content": "speaker_3"}, {"start": 79.6, "end": 79.68, "content": "speaker_3"}, {"start": 79.76, "end": 79.84, "content": "speaker_3"}, {"start": 86.56, "end": 87.44, "content": "speaker_3"}]

In [195]:
filename = 'katie_clipped_5-10min'
filepath = f'../data/audio/CHILDES/{filename}.wav'
sr = 16000
audio, _ = librosa.load(filepath, sr=sr)

In [196]:
# Group segments by speaker
speaker_segments = {}
for seg in data:
    speaker = seg["content"]
    speaker_segments.setdefault(speaker, []).append((seg["start"], seg["end"]))

In [197]:
# For each speaker, extract and concatenate their segments
speaker_audios = {}
for speaker, segments in speaker_segments.items():
    chunks = []
    for start, end in segments:
        start_sample = int(start * sr)
        end_sample = int(end * sr)
        chunks.append(audio[start_sample:end_sample])
    speaker_audios[speaker] = np.concatenate(chunks)

In [198]:
# Store results in an array (ordered by speaker)
result_audios = [speaker_audios[speaker] for speaker in speaker_audios]

In [199]:
# Save each speaker's audio to a file
for speaker, aud in speaker_audios.items():
    sf.write(f'../data/audio/CHILDES/{filename}_{speaker}.wav', aud, sr)

In [163]:
## Use SageMaker age classifier inference to determine if each speaker is adult or child

## Modifying speaker labels for child vs. adult using mapping obtained from age classifier

In [63]:
data = [{"start": 0.0, "end": 1.2, "content": "speaker_0"}, {"start": 2.4, "end": 3.52, "content": "speaker_0"}, {"start": 3.76, "end": 4.56, "content": "speaker_0"}, {"start": 4.64, "end": 5.6, "content": "speaker_0"}, {"start": 6.56, "end": 6.8, "content": "speaker_0"}, {"start": 8.32, "end": 9.92, "content": "speaker_0"}, {"start": 10.08, "end": 12.8, "content": "speaker_0"}, {"start": 13.52, "end": 13.68, "content": "speaker_0"}, {"start": 14.08, "end": 14.64, "content": "speaker_0"}, {"start": 14.96, "end": 17.76, "content": "speaker_0"}, {"start": 18.08, "end": 18.32, "content": "speaker_0"}, {"start": 18.48, "end": 19.92, "content": "speaker_0"}, {"start": 20.16, "end": 22.4, "content": "speaker_0"}, {"start": 22.88, "end": 24.72, "content": "speaker_0"}, {"start": 25.52, "end": 25.76, "content": "speaker_0"}, {"start": 26.72, "end": 27.6, "content": "speaker_0"}, {"start": 30.8, "end": 31.92, "content": "speaker_0"}, {"start": 32.72, "end": 33.04, "content": "speaker_0"}, {"start": 7.2, "end": 8.24, "content": "speaker_1"}, {"start": 37.68, "end": 40.64, "content": "speaker_1"}, {"start": 42.64, "end": 43.92, "content": "speaker_1"}, {"start": 44.32, "end": 44.88, "content": "speaker_1"}, {"start": 46.16, "end": 46.8, "content": "speaker_1"}, {"start": 51.6, "end": 53.2, "content": "speaker_1"}, {"start": 55.68, "end": 56.56, "content": "speaker_1"}, {"start": 57.52, "end": 59.68, "content": "speaker_1"}, {"start": 61.36, "end": 61.76, "content": "speaker_1"}, {"start": 61.92, "end": 62.8, "content": "speaker_1"}, {"start": 64.16, "end": 65.2, "content": "speaker_1"}, {"start": 66.64, "end": 66.8, "content": "speaker_1"}, {"start": 68.96, "end": 70.0, "content": "speaker_1"}, {"start": 70.8, "end": 72.16, "content": "speaker_1"}, {"start": 74.64, "end": 75.44, "content": "speaker_1"}, {"start": 76.32, "end": 77.52, "content": "speaker_1"}, {"start": 80.0, "end": 80.32, "content": "speaker_1"}, {"start": 84.24, "end": 84.48, "content": "speaker_1"}, {"start": 85.04, "end": 85.68, "content": "speaker_1"}, {"start": 88.0, "end": 89.2, "content": "speaker_1"}, {"start": 90.24, "end": 90.48, "content": "speaker_1"}, {"start": 91.92, "end": 92.32, "content": "speaker_1"}, {"start": 93.6, "end": 94.72, "content": "speaker_1"}, {"start": 96.4, "end": 97.04, "content": "speaker_1"}, {"start": 101.04, "end": 102.64, "content": "speaker_1"}, {"start": 103.84, "end": 104.24, "content": "speaker_1"}, {"start": 104.96, "end": 105.6, "content": "speaker_1"}, {"start": 107.04, "end": 108.32, "content": "speaker_1"}, {"start": 110.4, "end": 110.8, "content": "speaker_1"}, {"start": 111.28, "end": 113.52, "content": "speaker_1"}, {"start": 114.56, "end": 116.64, "content": "speaker_1"}, {"start": 118.0, "end": 118.48, "content": "speaker_1"}, {"start": 121.2, "end": 121.76, "content": "speaker_1"}, {"start": 123.36, "end": 124.64, "content": "speaker_1"}, {"start": 126.16, "end": 127.2, "content": "speaker_1"}, {"start": 129.04, "end": 130.56, "content": "speaker_1"}, {"start": 133.2, "end": 133.6, "content": "speaker_1"}, {"start": 140.08, "end": 141.44, "content": "speaker_1"}, {"start": 142.72, "end": 143.68, "content": "speaker_1"}, {"start": 145.52, "end": 147.2, "content": "speaker_1"}, {"start": 149.04, "end": 149.68, "content": "speaker_1"}, {"start": 151.76, "end": 152.4, "content": "speaker_1"}, {"start": 155.6, "end": 156.48, "content": "speaker_1"}, {"start": 159.04, "end": 159.28, "content": "speaker_1"}, {"start": 159.76, "end": 160.48, "content": "speaker_1"}, {"start": 162.24, "end": 163.68, "content": "speaker_1"}, {"start": 163.84, "end": 163.92, "content": "speaker_1"}, {"start": 164.16, "end": 164.56, "content": "speaker_1"}, {"start": 167.76, "end": 168.64, "content": "speaker_1"}, {"start": 170.16, "end": 170.48, "content": "speaker_1"}, {"start": 172.24, "end": 173.6, "content": "speaker_1"}, {"start": 179.6, "end": 180.96, "content": "speaker_1"}, {"start": 183.2, "end": 183.52, "content": "speaker_1"}, {"start": 187.76, "end": 187.84, "content": "speaker_1"}, {"start": 190.88, "end": 191.52, "content": "speaker_1"}, {"start": 194.56, "end": 195.2, "content": "speaker_1"}, {"start": 196.32, "end": 196.72, "content": "speaker_1"}, {"start": 200.88, "end": 201.44, "content": "speaker_1"}, {"start": 210.32, "end": 210.64, "content": "speaker_1"}, {"start": 212.0, "end": 213.28, "content": "speaker_1"}, {"start": 214.32, "end": 214.4, "content": "speaker_1"}, {"start": 214.8, "end": 216.08, "content": "speaker_1"}, {"start": 219.12, "end": 219.44, "content": "speaker_1"}, {"start": 220.16, "end": 221.44, "content": "speaker_1"}, {"start": 224.16, "end": 225.36, "content": "speaker_1"}, {"start": 232.56, "end": 234.4, "content": "speaker_1"}, {"start": 244.48, "end": 246.24, "content": "speaker_1"}, {"start": 246.32, "end": 247.12, "content": "speaker_1"}, {"start": 253.68, "end": 254.08, "content": "speaker_1"}, {"start": 254.16, "end": 255.92, "content": "speaker_1"}, {"start": 260.24, "end": 262.64, "content": "speaker_1"}, {"start": 263.92, "end": 265.28, "content": "speaker_1"}, {"start": 266.96, "end": 267.84, "content": "speaker_1"}, {"start": 269.04, "end": 270.8, "content": "speaker_1"}, {"start": 272.96, "end": 273.84, "content": "speaker_1"}, {"start": 275.04, "end": 275.76, "content": "speaker_1"}, {"start": 278.32, "end": 279.52, "content": "speaker_1"}, {"start": 282.56, "end": 283.04, "content": "speaker_1"}, {"start": 284.16, "end": 284.64, "content": "speaker_1"}, {"start": 285.92, "end": 286.24, "content": "speaker_1"}, {"start": 293.12, "end": 294.56, "content": "speaker_1"}, {"start": 294.8, "end": 295.12, "content": "speaker_1"}, {"start": 41.2, "end": 42.56, "content": "speaker_2"}, {"start": 44.72, "end": 45.28, "content": "speaker_2"}, {"start": 47.44, "end": 48.64, "content": "speaker_2"}, {"start": 51.2, "end": 51.44, "content": "speaker_2"}, {"start": 53.6, "end": 53.92, "content": "speaker_2"}, {"start": 54.0, "end": 55.2, "content": "speaker_2"}, {"start": 57.2, "end": 57.68, "content": "speaker_2"}, {"start": 60.72, "end": 61.2, "content": "speaker_2"}, {"start": 63.68, "end": 64.32, "content": "speaker_2"}, {"start": 65.52, "end": 66.24, "content": "speaker_2"}, {"start": 66.64, "end": 68.88, "content": "speaker_2"}, {"start": 73.2, "end": 74.32, "content": "speaker_2"}, {"start": 74.4, "end": 74.64, "content": "speaker_2"}, {"start": 75.92, "end": 76.32, "content": "speaker_2"}, {"start": 77.92, "end": 79.12, "content": "speaker_2"}, {"start": 79.44, "end": 79.76, "content": "speaker_2"}, {"start": 81.2, "end": 81.28, "content": "speaker_2"}, {"start": 86.0, "end": 86.56, "content": "speaker_2"}, {"start": 90.96, "end": 91.76, "content": "speaker_2"}, {"start": 101.36, "end": 102.0, "content": "speaker_2"}, {"start": 103.76, "end": 103.92, "content": "speaker_2"}, {"start": 104.08, "end": 104.8, "content": "speaker_2"}, {"start": 106.24, "end": 106.8, "content": "speaker_2"}, {"start": 108.8, "end": 110.48, "content": "speaker_2"}, {"start": 114.48, "end": 114.72, "content": "speaker_2"}, {"start": 116.56, "end": 117.76, "content": "speaker_2"}, {"start": 118.88, "end": 120.72, "content": "speaker_2"}, {"start": 122.16, "end": 123.44, "content": "speaker_2"}, {"start": 136.56, "end": 137.84, "content": "speaker_2"}, {"start": 138.16, "end": 139.52, "content": "speaker_2"}, {"start": 142.0, "end": 142.32, "content": "speaker_2"}, {"start": 144.0, "end": 144.32, "content": "speaker_2"}, {"start": 148.32, "end": 148.72, "content": "speaker_2"}, {"start": 149.84, "end": 151.6, "content": "speaker_2"}, {"start": 152.16, "end": 153.2, "content": "speaker_2"}, {"start": 157.84, "end": 158.88, "content": "speaker_2"}, {"start": 161.12, "end": 161.92, "content": "speaker_2"}, {"start": 169.2, "end": 169.6, "content": "speaker_2"}, {"start": 170.8, "end": 172.0, "content": "speaker_2"}, {"start": 174.4, "end": 175.36, "content": "speaker_2"}, {"start": 176.8, "end": 179.44, "content": "speaker_2"}, {"start": 181.76, "end": 182.72, "content": "speaker_2"}, {"start": 183.36, "end": 183.52, "content": "speaker_2"}, {"start": 183.84, "end": 185.2, "content": "speaker_2"}, {"start": 185.68, "end": 187.68, "content": "speaker_2"}, {"start": 188.4, "end": 189.36, "content": "speaker_2"}, {"start": 189.52, "end": 190.72, "content": "speaker_2"}, {"start": 195.68, "end": 195.84, "content": "speaker_2"}, {"start": 197.44, "end": 198.56, "content": "speaker_2"}, {"start": 201.68, "end": 202.56, "content": "speaker_2"}, {"start": 204.0, "end": 205.84, "content": "speaker_2"}, {"start": 206.72, "end": 208.32, "content": "speaker_2"}, {"start": 208.8, "end": 210.16, "content": "speaker_2"}, {"start": 213.68, "end": 214.24, "content": "speaker_2"}, {"start": 217.12, "end": 218.24, "content": "speaker_2"}, {"start": 222.16, "end": 222.56, "content": "speaker_2"}, {"start": 226.16, "end": 226.56, "content": "speaker_2"}, {"start": 227.92, "end": 230.16, "content": "speaker_2"}, {"start": 236.0, "end": 237.28, "content": "speaker_2"}, {"start": 239.92, "end": 241.76, "content": "speaker_2"}, {"start": 242.64, "end": 244.56, "content": "speaker_2"}, {"start": 247.76, "end": 248.96, "content": "speaker_2"}, {"start": 252.56, "end": 253.76, "content": "speaker_2"}, {"start": 257.52, "end": 258.4, "content": "speaker_2"}, {"start": 258.8, "end": 260.08, "content": "speaker_2"}, {"start": 268.8, "end": 269.36, "content": "speaker_2"}, {"start": 271.2, "end": 272.8, "content": "speaker_2"}, {"start": 276.08, "end": 276.88, "content": "speaker_2"}, {"start": 277.2, "end": 278.4, "content": "speaker_2"}, {"start": 280.08, "end": 281.92, "content": "speaker_2"}, {"start": 285.12, "end": 285.52, "content": "speaker_2"}, {"start": 287.36, "end": 289.12, "content": "speaker_2"}, {"start": 289.44, "end": 290.72, "content": "speaker_2"}, {"start": 291.68, "end": 292.56, "content": "speaker_2"}, {"start": 295.44, "end": 295.76, "content": "speaker_2"}, {"start": 60.4, "end": 60.72, "content": "speaker_3"}, {"start": 63.44, "end": 63.52, "content": "speaker_3"}, {"start": 74.24, "end": 74.4, "content": "speaker_3"}, {"start": 79.12, "end": 79.52, "content": "speaker_3"}, {"start": 79.6, "end": 79.68, "content": "speaker_3"}, {"start": 79.76, "end": 79.84, "content": "speaker_3"}, {"start": 86.56, "end": 87.44, "content": "speaker_3"}]

In [64]:
label_dict = {
  "speaker_0": "adult_0",
  "speaker_1": "adult_1",
  "speaker_2": "child_2",
  "speaker_3": "child_3",
}

In [65]:
def update_labels(data, label_dict):
  for item in data:
        item["content"] = label_dict[item["content"]]
  return data

In [66]:
new_data = update_labels(data, label_dict)

In [67]:
print(json.dumps(new_data))

[{"start": 0.0, "end": 1.2, "content": "adult_0"}, {"start": 2.4, "end": 3.52, "content": "adult_0"}, {"start": 3.76, "end": 4.56, "content": "adult_0"}, {"start": 4.64, "end": 5.6, "content": "adult_0"}, {"start": 6.56, "end": 6.8, "content": "adult_0"}, {"start": 8.32, "end": 9.92, "content": "adult_0"}, {"start": 10.08, "end": 12.8, "content": "adult_0"}, {"start": 13.52, "end": 13.68, "content": "adult_0"}, {"start": 14.08, "end": 14.64, "content": "adult_0"}, {"start": 14.96, "end": 17.76, "content": "adult_0"}, {"start": 18.08, "end": 18.32, "content": "adult_0"}, {"start": 18.48, "end": 19.92, "content": "adult_0"}, {"start": 20.16, "end": 22.4, "content": "adult_0"}, {"start": 22.88, "end": 24.72, "content": "adult_0"}, {"start": 25.52, "end": 25.76, "content": "adult_0"}, {"start": 26.72, "end": 27.6, "content": "adult_0"}, {"start": 30.8, "end": 31.92, "content": "adult_0"}, {"start": 32.72, "end": 33.04, "content": "adult_0"}, {"start": 7.2, "end": 8.24, "content": "adult_1"